## Simulation Option 1: Basic using only OpenMM

In [ ]:
# This script was generated by OpenMM-Setup on 2026-10-16.

from openmm import * # import OpenMM
# use the following to access the application layer of the Python API
from openmm.app import *
from openmm.unit import *

# Input Files

pdbx = PDBxFile('/n/home00/ewang03/lab/GroEL/results/openMM/DHFR_WT_apo.cif') # load PDB file of protein generated by OpenMM setup
forcefield = ForceField('amber19-all.xml', 'amber19/tip3pfb.xml') # load force fields

# System Configuration

nonbondedMethod = PME # method for how to treat nonbonded atoms
nonbondedCutoff = 1.0*nanometers # cutoff distance for nonbonded atoms
constraints = HBonds # additional H bond constraint
rigidWater = True # keep water rigid so water doesn't vibrate
constraintTolerance = 0.000001
#hydrogenMass = 1.5*amu
ewaldErrorTolerance = 0.0005

# Integration Options

dt = 0.002*picoseconds # want dt to be slower than the fastest fibration (1 fs for H bond)
temperature = 300*kelvin
friction = 1.0/picosecond # friction between solvent molecules
pressure = 1.0*atmospheres
barostatInterval = 25 # readjust pressure every 25 steps

# Simulation Options

steps = 20000
equilibrationSteps = 1000
# StateDataReporter prints some basic info about the state of the sim to a file during the run
dataReporter = StateDataReporter('log.txt', 100, totalSteps=steps,
    step=True, speed=True, progress=True, potentialEnergy=True, temperature=True, density=True, separator='\t') # report anything that's set to true, always print step and potentialEnergy, separator is comma or tab

# Prepare the Simulation

print('Building system...')
topology = pdbx.topology # topology contains information about the identity and connectivity of particles in the system (i.e., chains and residues)
positions = pdbx.positions # contains coordinates of particles
system = forcefield.createSystem(topology, nonbondedMethod=nonbondedMethod, nonbondedCutoff=nonbondedCutoff,
    constraints=constraints, rigidWater=rigidWater, ewaldErrorTolerance=ewaldErrorTolerance, hydrogenMass=hydrogenMass)
system.addForce(MonteCarloBarostat(pressure, temperature, barostatInterval)) # control pressure and temp. add additional forces
integrator = LangevinMiddleIntegrator(temperature, friction, dt) # integration algorithm, defines how you go from one step to the next
integrator.setConstraintTolerance(constraintTolerance)
simulation = Simulation(topology, system, integrator) # create Simulation object to manage the simulation
simulation.context.setPositions(positions) # context object contains info about state of simulation, including particle positions and velocities

# Minimize and Equilibrate

print('Performing energy minimization...')
simulation.minimizeEnergy()
print('Equilibrating...')
simulation.context.setVelocitiesToTemperature(temperature)
simulation.step(equilibrationSteps)

# Simulate

print('Simulating...')
simulation.reporters.append(dataReporter)
simulation.currentStep = 0
simulation.step(steps)

Building system...
Performing energy minimization...
Equilibrating...
Simulating...


In [ ]:
# plots for simulation output
import numpy as np
import matplotlib.pyplot as plt

step, pe, temp, rho = np.loadtxt("log.txt", delimiter="\t", usecols=(1, 2, 3, 4)).T
time = step * dt.value_in_unit(picosecond)

figure = plt.figure(figsize=(8, 2.5), layout="constrained")

axes_pe = figure.add_subplot(1, 3, 1)
axes_pe.plot(time, pe)
axes_pe.set_xlabel("Time (ps)")
axes_pe.set_ylabel("Potential energy (kJ/mol)")

axes_temp = figure.add_subplot(1, 3, 2)
axes_temp.plot(time, temp)
axes_temp.set_xlabel("Time (ps)")
axes_temp.set_ylabel("Temperature (K)")

axes_rho = figure.add_subplot(1, 3, 3)
axes_rho.plot(time, rho)
axes_rho.set_xlabel("Time (ps)")
axes_rho.set_ylabel("Density (g/mL)")

plt.show()

## Simulation Option 2: Use external tool for molecule set up

In [ ]:
# This script was generated by OpenMM-Setup on 2025-09-25.

# import modules
from openmm import *
from openmm.app import *
from openmm.unit import *

# Input Files

inpcrd = AmberInpcrdFile('input.inpcrd')
prmtop = AmberPrmtopFile('input.prmtop', periodicBoxVectors=inpcrd.boxVectors)

# System Configuration

nonbondedMethod = PME
nonbondedCutoff = 1.0*nanometers
constraints = HBonds
rigidWater = True
constraintTolerance = 0.000001
hydrogenMass = 1.5*amu
ewaldErrorTolerance = 0.0005

# Integration Options

dt = 0.004*picoseconds
temperature = 300*kelvin
friction = 1.0/picosecond
pressure = 1.0*atmospheres
barostatInterval = 25

# Simulation Options

steps = 100000
equilibrationSteps = 10000
dcdReporter = DCDReporter('trajectory.dcd', 10000) # write coordinates in DCD format used by CHARMM and NAMD
dataReporter = StateDataReporter('log.txt', 1000, totalSteps=steps,
    step=True, speed=True, progress=True, potentialEnergy=True, temperature=True, separator='\t')
checkpointReporter = CheckpointReporter('checkpoint.chk', 10000) # save the current state of the sim periodically. Update every 10,000 steps

# Prepare the Simulation

print('Building system...')
topology = prmtop.topology
positions = inpcrd.positions
system = prmtop.createSystem(nonbondedMethod=nonbondedMethod, nonbondedCutoff=nonbondedCutoff,
    constraints=constraints, rigidWater=rigidWater, ewaldErrorTolerance=ewaldErrorTolerance, hydrogenMass=hydrogenMass)
system.addForce(MonteCarloBarostat(pressure, temperature, barostatInterval))
integrator = LangevinMiddleIntegrator(temperature, friction, dt)
integrator.setConstraintTolerance(constraintTolerance)
simulation = Simulation(topology, system, integrator) # create sim
simulation.context.setPositions(positions) # initialize positions and velocities

# Minimize and Equilibrate

print('Performing energy minimization...')
simulation.minimizeEnergy()
print('Equilibrating...')
simulation.context.setVelocitiesToTemperature(temperature)
simulation.step(equilibrationSteps)

# Simulate

print('Simulating...')
simulation.reporters.append(dcdReporter)
simulation.reporters.append(dataReporter)
simulation.reporters.append(checkpointReporter)
simulation.currentStep = 0
simulation.step(steps)

In [ ]:
# Customizing output

state = simulation.context.getState(positions=True, forces=True) # get positions of all atoms and the forces on them
                                                                 # make sure to set item = True for any data you want to retrieve (e.g., for velocity, must set velocities = True here to call it when getting data)
state.getPositions(asNumpy=True) # retrieve position data
state.getForces(asNumpy=True) # retrieve force data

In [ ]:
# writing your own reporter
# compute center of mass
def compute_com_nm(state, indices, masses):
    positions = state.getPositions(asNumpy=True).value_in_unit(nanometer) # Quantity.value_in_unit generates the values in those units
    return (masses[:, None] * positions[indices]).sum(axis=0) / masses.sum()

In [ ]:
# create custom COM reporter
import numpy as np

class COMReporter:
    def __init__(self, file, reportInterval, indices):
        # OpenMM does not manage the output file(s); reporters have full control over this.
        self._out = open(file, 'w')
        self._reportInterval = reportInterval

        # Save the indices given and look up the atom masses with System.getParticleMass().
        self._indices = np.array(indices)
        self._masses = np.array([system.getParticleMass(index).value_in_unit(dalton) for index in self._indices])

    def __del__(self):
        self._out.close()

    def describeNextReport(self, simulation):
        # Calculate the number of steps to the next report, if reports should happen at multiples of reportInterval steps.
        steps = self._reportInterval - simulation.currentStep % self._reportInterval
        return dict(steps=steps, include=['positions'], periodic=False)

    def report(self, simulation, state):
        x, y, z = compute_com_nm(state, self._indices, self._masses)

        # Do any kind of reporting.  We flush the file to make sure that a new line is written immediately.
        print(f"{x:12.6f} {y:12.6f} {z:12.6f}", file=self._out)
        self._out.flush()

In [ ]:
# use reporter
# retrieve indices of residues
residues = list(topology.residues())[:3]
residues

indices = [atom.index for residue in residues for atom in residue.atoms()]
indices

# create instance of custom reporter and run
simulation.reporters.append(COMReporter("com.txt", 100, indices))
simulation.step(steps)